# Корректирующее обучение FastSpeech 2

Цель: улучшить свободное предсказание duration, pitch и energy, не изменяя уже обученный mel-decoder и не перезаписывая результаты лабораторной №4.

## 1. Почему требуется fine-tuning

HiFi-GAN от эталонной mel звучит хорошо. На полном test split старая модель имеет duration MAE 3.554 кадра, duration correlation 0.427, pitch correlation 0.224, energy correlation 0.278 и в среднем создаёт только 73% нужных кадров. Поэтому исправляем variance predictors, сохраняя encoder и decoder замороженными.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

def locate_repo_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'labs/lab4_acoustic').is_dir() and (candidate / 'data/lab4_acoustic').is_dir():
            return candidate
    raise FileNotFoundError('Не найден корень репозитория')

REPO_ROOT = locate_repo_root()
for path in (REPO_ROOT, REPO_ROOT / 'labs/lab4_acoustic'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from acoustic_data import create_or_load_splits
from labs.lab4_acoustic_repair.repair_training import (
    initialize_repair_training, select_best_checkpoint, select_smoke_ids,
)
from labs.lab4_acoustic_repair.evaluate_variance_predictors import evaluate

splits, split_meta = create_or_load_splits(REPO_ROOT)
{name: len(frame) for name, frame in splits.items()}

## 2. Параметры запуска

Сначала используйте `smoke`. Для нового запуска оставьте `CONTINUE_PATH = None`. Для восстановления прерванного запуска вставьте путь к его каталогу.

In [ ]:
MODE = 'smoke'                 # 'smoke' или 'full'
RUN_TRAINING = False           # защитный переключатель
RUN_EVALUATION = False         # включить после появления best_model
RUN_AUDIO_DIAGNOSTIC = False   # создать WAV только после оценки

SOURCE_CHECKPOINT = REPO_ROOT / 'labs/lab4_acoustic/artifacts/best_model_742028.pth'
CONTINUE_PATH = None           # r'C:\\...\\ruslan_fastspeech2_variance_repair_full-...'
BATCH_SIZE = 8
NUM_WORKERS = 0                # на Windows начните с 0; затем можно 2–4
LEARNING_RATE = 1e-4
EPOCHS = 2 if MODE == 'smoke' else 15

selected_ids = select_smoke_ids(splits) if MODE == 'smoke' else None
print({'mode': MODE, 'epochs': EPOCHS, 'selected': None if selected_ids is None else len(selected_ids)})

## 3. Инициализация и обязательная проверка

Для нового run загружаются только веса исходного checkpoint. Старые optimizer и scheduler не используются.

In [ ]:
trainer, config, tokenizer, details = initialize_repair_training(
    REPO_ROOT,
    splits,
    source_checkpoint=SOURCE_CHECKPOINT,
    mode=MODE,
    selected_ids=selected_ids,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    num_workers=NUM_WORKERS,
    learning_rate=LEARNING_RATE,
    continue_path=CONTINUE_PATH,
)
summary = dict(details)
summary['parameter_summary'] = {
    key: value for key, value in details['parameter_summary'].items() if key != 'trainable_names'
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
trainable_names = details['parameter_summary']['trainable_names']
assert trainable_names
assert all(name.startswith(('duration_predictor.', 'pitch_predictor.', 'energy_predictor.')) for name in trainable_names)
assert config.spec_loss_alpha == 0.0
assert config.dur_loss_alpha == config.pitch_loss_alpha == config.energy_loss_alpha == 1.0
assert config.lr_scheduler is None
print('Проверка пройдена: обучаются только три predictor-блока, decoder заморожен.')

## 4. Fine-tuning

Обучение начнётся только после явного переключения `RUN_TRAINING = True`. Не запускайте одновременно старый training notebook.

In [ ]:
if RUN_TRAINING:
    trainer.fit()
else:
    print('Обучение отключено. Проверьте summary и установите RUN_TRAINING = True.')

## 5. Оценка кандидата

Оценка использует неизменявшийся test split и моделирует свободные duration, pitch и energy. Test нельзя использовать для выбора эпохи; эпоха выбирается по validation loss, test запускается один раз для принятого кандидата.

In [ ]:
candidate_checkpoint = None
if RUN_EVALUATION:
    run_dir = Path(CONTINUE_PATH).resolve() if CONTINUE_PATH else Path(details['output_path'])
    candidate_checkpoint = select_best_checkpoint(run_dir)
    candidate_metrics = evaluate(candidate_checkpoint, batch_size=16, num_workers=NUM_WORKERS)
    metrics_path = REPO_ROOT / 'data/lab4_acoustic_repair/candidate_metrics.json'
    metrics_path.parent.mkdir(parents=True, exist_ok=True)
    metrics_path.write_text(json.dumps(candidate_metrics, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print(json.dumps(candidate_metrics, ensure_ascii=False, indent=2))
else:
    print('Оценка отключена. Включите её только после завершения обучения.')

In [ ]:
if RUN_AUDIO_DIAGNOSTIC:
    if candidate_checkpoint is None:
        raise RuntimeError('Сначала выполните оценку кандидата')
    subprocess.run([
        sys.executable, '-m', 'labs.lab5_service.diagnose_acoustic_model',
        '--checkpoint', str(candidate_checkpoint), '--write-audio',
    ], cwd=REPO_ROOT, check=True)
else:
    print('Аудиодиагностика отключена.')

## 6. Критерии принятия

Кандидат должен одновременно улучшить duration/pitch/energy относительно baseline, приблизить median frame ratio к 1.0 и дать разборчивый свободный WAV. Хороший teacher-forced WAV сам по себе недостаточен. До ручного прослушивания checkpoint не переносится ни в исходные artifacts лабораторной №4, ни в сервис лабораторной №5.